DIY Disease Tracking Dashboard

In [1]:
from IPython.display import clear_output
import ipywidgets as wdg
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import time
import json

In [2]:
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [3]:
with open("admissions_rate.json", "rt") as INFILE:
    admissions_rate = json.load(INFILE)

with open("ICUHDUadmissions_rate.json", "rt") as INFILE:
    icuhdu_admissions_rate = json.load(INFILE)

In [4]:
def wrangle_data(rawdata):
    data = {}
    for dataset_name, dataset in rawdata.items():
        for entry in dataset:
            date = entry['date']
            value = entry['metric_value']
            if 'ICU' in entry['metric']:
                key = 'ICU HDU admissions'
            else:
                key = 'admissions'
            
            if date not in data:
                data[date] = {}
            data[date][key] = value

    df = pd.DataFrame.from_dict(data, orient='index')
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True)
    df.fillna(0.0, inplace=True)
    return df

# putting the wrangling code into a function allows you to call it again after refreshing the data through 
# the API. You should call the function directly on the JSON data when the dashboard starts, by including 
# the call in this cell as below:
timeseriesdf = wrangle_data({
    'admissions_rate': admissions_rate,
    'ICUHDUadmissions_rate': icuhdu_admissions_rate
})

In [5]:
def access_api():
    """Access the UKHSA API and return data for both hospital and ICU admissions."""
    
    structure = {
        "theme": "infectious_disease",
        "sub_theme": "respiratory",
        "topic": "Influenza",
        "geography_type": "Nation",
        "geography": "England"
    }

    filters = {
        "year": 2022,
        "stratum": None,
        "age": None,
        "sex": None,
        "month": None,
        "epiweek": None,
        "date": None,
        "in_reporting_delay_period": None
    }

    # --- First metric: Hospital admission rate ---
    structure["metric"] = "influenza_healthcare_hospitalAdmissionRateByWeek"
    api = APIwrapper(**structure)
    admissions_rate = api.get_all_pages(filters)
    print(f"Hospital data: {len(admissions_rate)} points retrieved.")

    # --- Second metric: ICU/HDU admission rate ---
    structure["metric"] = "influenza_healthcare_ICUHDUadmissionRateByWeek"
    api = APIwrapper(**structure)
    icuhdu_admissions_rate = api.get_all_pages(filters)
    print(f"ICU/HDU data: {len(icuhdu_admissions_rate)} points retrieved.")

    # Return both datasets as a dictionary
    return {
        "admissions_rate": admissions_rate,
        "ICUHDUadmissions_rate": icuhdu_admissions_rate
    }

In [8]:
def api_button_callback(button):
    """ Button callback - it must take the button as its parameter (unused in this case).
    Accesses API, wrangles data, updates global variable df used for plotting. """
    # Get fresh data from the API. If you have time, include some error handling
    # around this call.
    apidata=access_api()
    # wrangle the data and overwrite the dataframe for plotting
    global df
    df=wrangle_data(apidata)
    # the graph won't refresh until the user interacts with the widget.
    # this function simulates the interaction, see Graph and Analysis below.
    # The function needs to be adapted to your graph; you can omit this call
    # in the first instance
    refresh_graph()
    # after all is done, you can switch the icon on the button to a "check" sign
    # and optionally disable the button - it won't be needed again. If you are 
    # implementing error handling, you can use icons "unlink" or "times" and 
    # change the button text to "Unavailable" when the api call fails.
    apibutton.icon="check"
    # apibutton.disabled=True

#pulls data from API then codes the wrangle function, transforms into dataframe.
    
apibutton = wdg.Button(
    description='Update data',  # clearer and friendlier label
    disabled=False,
    button_style='info',  # blue = neutral action
    tooltip="Click to fetch the latest influenza data from UKHSA",
    icon='sync'  # circular arrows = refresh
)

# remember to register your button callback function with the button
apibutton.on_click(api_button_callback) # the name of your function inside these brackets

display(apibutton)

# run all cells before clicking on this button

Button(button_style='info', description='Update data', icon='sync', style=ButtonStyle(), tooltip='Click to fet…

In [9]:
series=wdg.SelectMultiple(
    options=['admissions', 'ICU HDU admissions'],
    value=['admissions', 'ICU HDU admissions'],
    rows=2,
    description='Stats:',
    disabled=False
)

scale=wdg.RadioButtons(
    options=['linear', 'log'],
#   value='pineapple', # Defaults to 'pineapple'
#   layout={'width': 'max-content'}, # If the items' names are long
    description='Scale:',
    disabled=False
)

controls=wdg.HBox([series, scale])

def timeseries_graph(gcols, gscale):
    if gscale=='linear':
        logscale=False
    else:
        logscale=True
    ncols=len(gcols)
    if ncols>0:
        timeseriesdf[list(gcols)].plot(logy=logscale)
        plt.show() # important - graphs won't update if this is missing 
    else:
        print("Click to select data for graph")
        print("(CTRL-Click to select more than one category)")
 
graph=wdg.interactive_output(timeseries_graph, {'gcols': series, 'gscale': scale})

display(controls, graph)

Output()